# 第8回A 演習: 言語モデルの仕組みと実装

スライド: [第8回A](../08a-llm/)

言語モデル (文章の続きを予測するプログラム) を使って文章を作り、最後に簡単なチャットを作る。
モデルを動かす部分 (`encode`, `next_scores`) は用意してある。
自分で書くのは普通の Python (リスト・`math`・`random`) だけで、PyTorch は使わない。

## このノートブックの使い方

1. Colab で `ランタイム → ランタイムのタイプを変更 → T4 GPU` を選ぶ (CPU でも動くが、生成が遅い)
2. 上のセルから順に実行する。最初のモデル読み込みに数分かかるので、待つ間に課題を読む
3. 自分で書く場所は `TODO` の行と、`...` や `pass` の所。消して自分のコードに置き換える
4. 各課題の下の確認セル (`# [CHECK]` で始まるセル) を実行する。`OK` と出れば正解。確認セルは書き換えない
5. `NameError` (名前が見つからない) と出たら、上のセルを実行していない。`ランタイム → 前のセルを実行`
6. Colab を開き直したときや、接続が切れたときは、上から全部実行し直す

## 言葉の説明

- **トークン**: モデルが扱う文字列の小さな単位。単語のことも、1文字や文字の一部のこともある
- **トークン ID**: トークンに付いた番号 (整数)。モデルは文字列ではなく、この番号の列を受け取る
- **スコア**: モデルが次の候補それぞれに付ける点数。大きいほど次に来やすい。確率ではない
- **確率**: 0 から 1 の数で、全部足すと 1 になる

## 課題の一覧

| 課題 | 配点 |
|---|---|
| 課題1 スコアを確率にする (softmax) | 10 |
| 課題1-b 次のトークンの上位候補 | 10 |
| 課題2 greedy 生成 | 20 |
| 課題2-b 繰り返し率 | 5 |
| 課題3 温度と top-k / top-p | 20 |
| 課題4 chat | 10 |
| 課題4-b 会話を続ける | 5 |
| 課題5 簡易チャットアプリ | 出力で確認 |
| 発展1〜4 (繰り返し検出 / 圧縮 / ストリーミング / エフォート) | 各5 |


---

## セットアップ

下の3つのセルを上から実行する。モデルの取得に数分かかる。

1. **ライブラリ更新**: このモデルには新しい transformers が必要。赤い警告が出ても、エラーで止まらなければよい
2. **モデル読み込み**: Qwen3.5-0.8B を読み込み、`encode` と `next_scores` を用意する。最後に数字のリストが出れば成功
3. **自動セットアップ**: 確認セルを動かす準備

「セッションを再起動」を求められたら再起動し、もう一度このセルから実行する。


In [ ]:
# Qwen3.5 は最新の transformers が必要
!pip install -q -U transformers

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText

MODEL = 'Qwen/Qwen3.5-0.8B'  # マルチモーダル。テキストのみで使う

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForImageTextToText.from_pretrained(MODEL)
model.eval()

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
model.to(device)
print(device, '/ 語彙数:', len(tok))

# ---- 以降で使うヘルパ (中身は PyTorch。読まなくてよい) ----
def encode(text):
    """文字列 -> トークン ID のリスト"""
    return tok(text).input_ids


def next_scores(ids, n=100):
    """トークン ID のリストから、次に来るトークンの候補を返す。

    [(トークンID, スコア), ...] を、スコアの高い順に n 件。
    スコアは確率ではない。softmax を通すと確率になる (課題1)。
    """
    with torch.no_grad():
        logits = model(torch.tensor([ids], device=device)).logits[0, -1].float()
    top = torch.topk(logits, n)
    return list(zip(top.indices.tolist(), top.values.tolist()))


print(encode('こんにちは'))
print(next_scores(encode('こんにちは'))[:3])

In [ ]:
#@title 自動セットアップ (実行するだけ。中身は隠しています)
# [SKIP] 採点の対象外
# ---- 自己確認の準備 (check / score)。1回実行すればよい ----
import pathlib
import urllib.request

if not pathlib.Path('autograde.py').exists():
    try:
        urllib.request.urlretrieve(
            'https://3tty0n.github.io/2026-scripting-lab/grading/autograde.py', 'autograde.py')
    except Exception as e:
        print('取得できませんでした:', e)

from autograde import check, score
TASK = '08a'


---
## 課題1: スコアを確率にする (10点)

**やること**: スコアのリストを受け取り、確率のリスト (どれも 0〜1、合計 1) を返す
`softmax(scores, temperature=1.0)` を書く。softmax はこの変換の名前。

**入出力の例**

```
softmax([1.0, 1.0])        → [0.5, 0.5]
softmax([1.0, 2.0, 3.0])   → [0.09, 0.24, 0.67]
softmax([3.0, 1.0], 2.0)   → [0.73, 0.27]
```

**手順** (コードの `TODO 1〜3` と同じ順)

1. 各スコアを温度 `temperature` で割る
2. その中の最大値を引いてから、`math.exp` を取る
3. 全体の合計で割ったリストを返す

**ヒント**: 温度は「確率の差の付け方」を変える数。小さいほど1位に偏り、大きいほどばらける。
最大値を引くのは、`math.exp` に大きな数を入れると `OverflowError` (数が大きすぎる) になるのを防ぐため。
引いても確率は変わらない。

**つまずきやすい点**

- 確認セルで「あふれた」と出る → 手順2 で最大値を引いていない
- 「合計が 1 になっていない」→ 手順3 で全体の合計で割っていない
- 温度の確認 (`0.1` や `2.0`) で止まる → 手順1 で温度で割っていない

### 例題

次のセルを実行して、動きを見る。
`next_scores` が返す「(トークンID, スコア) のリスト」と、`exp` で割合にする流れ。

In [ ]:
# 例題
import math

scores = next_scores(encode('日本の首都は'))
print(scores[:3])    # [(100562, 22.4), (3583, 20.9), ...]

xs = [1.0, 2.0, 3.0]
es = [math.exp(x) for x in xs]
print([e / sum(es) for e in es])    # [0.09, 0.24, 0.67]

温度で割ってから `exp` を取ると、温度が低いほど1位の確率が大きくなる。
`softmax([3.0, 1.0], 0.1)` はほぼ `[1.0, 0.0]`、`softmax([3.0, 1.0], 2.0)` は `[0.73, 0.27]`。

In [ ]:
import math


def softmax(scores, temperature=1.0):
    # TODO 1: 各スコアを temperature で割ったリストを作る
    # TODO 2: 最大値を引いてから math.exp を取る
    # TODO 3: 合計で割ったリストを return する
    pass


print(softmax([1.0, 2.0, 3.0]))
print(softmax([3.0, 1.0], 0.1))
print(softmax([3.0, 1.0], 2.0))

In [ ]:
# [CHECK] q1
need('softmax', arity=2)

p = softmax([1.0, 1.0])
assert len(p) == 2 and all(abs(x - 0.5) < 1e-9 for x in p), p

p = softmax([1.0, 2.0, 3.0])
assert abs(sum(p) - 1.0) < 1e-9, '合計が 1 になっていない'
assert p == sorted(p), '大きいスコアほど大きい確率にすること'
assert abs(p[2] - 0.6652409557748219) < 1e-9, p

# 温度: 低いと決め打ち、高いとばらける
assert softmax([3.0, 1.0], 0.1)[0] > 0.99
assert abs(softmax([3.0, 1.0], 2.0)[0] - 0.7310585786300049) < 1e-9

# 大きいスコアでもあふれない
p = softmax([1000.0, 1000.0])
assert abs(p[0] - 0.5) < 1e-9, 'あふれた (最大値を引いてから exp を取る)'
print('OK')

---
### 課題1-b: 次のトークンの候補を見る (10点)

**やること**: 文字列 `text` の次に来そうなトークンを、上位 `n` 個だけ
`[(トークンの文字列, 確率), ...]` の形で返す `top_candidates(text, n=10)` を書く。

**入出力の例** (数字は一例)

```
top_candidates('日本の首都は', 3) → [('東京', 0.62), ('、', 0.10), ('日本', 0.05)]
```

**手順**

1. `next_scores(encode(text))` で候補 `(ID, スコア)` のリストを得る
2. スコアだけを取り出して、`softmax` で確率にする
3. 先頭 `n` 件を `(tok.decode([ID]), 確率)` の組にして、リストで返す

**ヒント**: `next_scores` は最初から高い順なので、並べ替えは要らない。
確率は `next_scores` が返す100件の中での割合でよい。

**つまずきやすい点**

- 「トークンは文字列で返すこと」→ ID のまま返している。`tok.decode([ID])` で文字列にする (`[ ]` で囲んで渡す)
- 「確率の合計が1を超えている」→ `softmax` を通していない
- 「上位1件が最高スコアの候補と違う」→ 並べ替えて順番を変えている

### 例題

次のセルを実行して、動きを見る。
`(ID, スコア)` の組のリストから、ID とスコアをそれぞれ取り出す。

In [ ]:
# 例題
scores = next_scores(encode('日本の首都は'))
ids = [i for i, _ in scores]
vals = [s for _, s in scores]
print(ids[:3], vals[:3])
print(repr(tok.decode([ids[0]])))    # '東京'

あとは `softmax(vals)` で確率にして、先頭 `n` 件を `(文字列, 確率)` に組み立てる。
`zip(ids, probs)` で ID と確率を対にできる。

In [ ]:
def top_candidates(text, n=10):
    # TODO 1: next_scores(encode(text)) で候補を得る
    # TODO 2: スコアだけのリストを softmax で確率にする
    # TODO 3: 先頭 n 件を (tok.decode([ID]), 確率) のリストにして return する
    pass


for t, p in top_candidates('むかしむかし、あるところに'):
    print(f'{t!r:20} {p:.4f}')

In [ ]:
# [CHECK] q1b
need('top_candidates', arity=2)

cand = top_candidates('日本の首都は', n=10)
assert isinstance(cand, list) and len(cand) == 10, len(cand)
assert all(isinstance(t, str) for t, _ in cand), 'トークンは文字列で返すこと'
probs = [float(p) for _, p in cand]
assert probs == sorted(probs, reverse=True), '確率の降順になっていない'
assert 0 < probs[0] <= 1.0, probs[0]
assert sum(probs) <= 1.0001, '確率の合計が1を超えている (softmax を通したか)'
assert len(top_candidates('こんにちは', n=3)) == 3

# 上位1件は、最高スコアの候補と一致するはず
top_id = next_scores(encode('日本の首都は'))[0][0]
assert cand[0][0] == tok.decode([top_id]), '上位1件が最高スコアの候補と違う'
print('OK')

---
## 課題2: greedy 生成 (20点)

**やること**: 毎回いちばんスコアの高いトークンを選んで、文章を伸ばしていく
`generate_greedy(prompt, max_new_tokens)` を書く。
greedy (貪欲法) は「その時点で一番よいものを選ぶ」やり方。同じ入力なら毎回同じ結果になる。

**入出力の例** (文章は一例)

```
generate_greedy('日本で一番高い山は', 20) → '日本で一番高い山は富士山です。...'
```

**手順** (ループの中身を書く。ループと最後の `return` は用意してある)

1. `scores[0][0]` (いちばん高い候補の ID) を `next_id` にする
2. `next_id` が終わりの印 `tok.eos_token_id` なら、`break` でループを抜ける
3. そうでなければ、`ids.append(next_id)` で末尾に足す

**ヒント**: `max_new_tokens` は「最大で何トークン足すか」。ループの回数はこれで決まっている。

**つまずきやすい点**

- 「何も生成されていない」→ 手順3 の `append` が抜けている
- 「… トークン生成された」→ ループの1回で2つ以上足している
- 返り値の先頭が入力と違う → `ids` を最初から作り直している。`ids` には足すだけにする

### 例題

次のセルを実行して、動きを見る。
1歩だけ進めてみる。ループはこれの繰り返し。

In [ ]:
# 例題
ids = encode('日本で一番高い山は')
print(ids)    # [101293, ...]

scores = next_scores(ids)
next_id = scores[0][0]
print(next_id, repr(tok.decode([next_id])))

ids.append(next_id)
print(len(ids))
print(tok.decode(ids, skip_special_tokens=True))

終わりの判定は `if next_id == tok.eos_token_id: break`。
`ids` に足す**前**に置くこと。

In [ ]:
def generate_greedy(prompt, max_new_tokens=40):
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        scores = next_scores(ids)
        # TODO 1: next_id = scores[0][0] (いちばん高い候補の ID)
        # TODO 2: next_id が tok.eos_token_id なら break
        # TODO 3: ids の末尾に next_id を足す
        pass
    return tok.decode(ids, skip_special_tokens=True)


print(generate_greedy('むかしむかし、あるところに'))

In [ ]:
# [CHECK] q2
need('generate_greedy', arity=2)

a = generate_greedy('日本で一番高い山は', 20)
b = generate_greedy('日本で一番高い山は', 20)
assert a == b, 'greedy なのに結果が変わっている'
assert a.startswith('日本で一番高い山は'), a[:30]
assert len(a) > len('日本で一番高い山は'), '何も生成されていない'

# max_new_tokens を守ること
short = generate_greedy('むかしむかし', 3)
n_new = len(tok(short).input_ids) - len(tok('むかしむかし').input_ids)
assert n_new <= 3, f'{n_new} トークン生成された'
assert generate_greedy('あ', 0) == 'あ'
print('OK')


### 課題2-b: 繰り返し率を測る (5点)

greedy は同じ言い回しをくり返しやすい。それを数で測る。

**やること**: 文章の中で、同じ言い回しがどのくらいくり返されているかを、0〜1 の数で返す
`repetition_rate(text, n=3)` を書く。
空白で区切った単語を、連続した `n` 個ずつのまとまり (n-gram と呼ぶ) にする。
そのうち、2回以上出てくるまとまりが全体の何割かを返す。

**入出力の例**

```
repetition_rate('a b c a b c a b c', 3) → 1.0    (7個のまとまりが、全部2回以上出る)
repetition_rate('x y z x y z p q r', 3) → 0.2857 (7個のうち、(x, y, z) の2個だけ)
repetition_rate('a b', 3)               → 0.0    (3個のまとまりが作れない)
```

**手順**

1. `text.split()` で単語のリストにする
2. `n` 個ずつのまとまり (タプル) のリストを作る
3. まとまりが1つもなければ `0.0` を返す
4. `Counter` で数え、「2回以上出るまとまりの個数 ÷ 全部の個数」を返す

**つまずきやすい点**

- 「n-gram が作れないときは 0.0」→ 手順3 が抜けている (0 で割るエラーになることもある)
- `2/7` の確認で止まる → 種類の数で割っている。重複も含めた全部の個数で割る

### 例題

次のセルを実行して、動きを見る。
まとまりの作り方と、数え方。

In [ ]:
# 例題
from collections import Counter

ws = 'a b c a b'.split()
grams = [tuple(ws[i:i + 2]) for i in range(len(ws) - 2 + 1)]
print(grams)          # [('a','b'), ('b','c'), ('c','a'), ('a','b')]
print(Counter(grams)) # Counter({('a','b'): 2, ('b','c'): 1, ('c','a'): 1})


あとは「`count >= 2` のまとまりが、全 4 個のうちいくつか」を数える。
この例なら 2/4 = 0.5。
`len(ws) - n + 1` が 0 以下のときは、まとまりが作れないので `0.0` を返す。

In [ ]:
from collections import Counter


def repetition_rate(text, n=3):
    # TODO 1: 単語のリストにする
    # TODO 2: n 個ずつのまとまり (タプル) のリストを作る
    # TODO 3: まとまりが無ければ 0.0 を返す
    # TODO 4: 2回以上出るまとまりの個数 / 全部の個数 を返す
    pass


# 実際に測ってみる
out = generate_greedy('好きな食べ物は', 80)
print(out)
print('繰り返し率:', repetition_rate(out))


In [ ]:
# [CHECK] q2b
need('repetition_rate', arity=2)

assert repetition_rate('a b c a b c a b c', n=3) == 1.0
assert repetition_rate('a b c d e f g', n=3) == 0.0
assert repetition_rate('', n=3) == 0.0
assert repetition_rate('a b', n=3) == 0.0, 'n-gram が作れないときは 0.0'
assert repetition_rate('a b c', n=3) == 0.0, '1個だけなら繰り返しではない'

r = repetition_rate('x y z x y z p q r', n=3)
assert 0.0 < r < 1.0, r
assert abs(r - 2/7) < 1e-9, f'{r} (7個の3-gram のうち (x,y,z) だけが2回)'

assert repetition_rate('a a a a', n=1) == 1.0
assert isinstance(repetition_rate('a b c a b c', n=2), float)

# 性質: 割合なので必ず 0 以上 1 以下
holds('0 以上 1 以下',
      lambda ws: 0.0 <= repetition_rate(' '.join(ws), 2) <= 1.0,
      [words(n) for n in range(0, 12)])
holds('同じ語だけ並べたら 1.0',
      lambda n: repetition_rate(' '.join(['x'] * n), 2) == 1.0, range(4, 12))
print('OK')


---
## 課題3: 温度と top-k / top-p (20点)

**やること**: 次のトークンを、確率に従ったくじ引きで1つ選ぶ関数を2つ書く。
どちらも返すのは**トークン ID (整数)**。`scores` は `next_scores` の返り値。

- `sample_next(scores, temperature, k)`: 上位 `k` 個だけを候補にする (top-k)
- `sample_next_top_p(scores, temperature, p)`: 確率の合計が `p` に届くまでの候補だけを残す (top-p)

greedy と違って毎回結果が変わるので、文章が単調になりにくい。

**入出力の例**

```
sample_next(scores, 1.0, 1)   → scores[0][0] と同じ (候補が1個しかない)
sample_next(scores, 0.8, 50)  → 上位50個のどれかの ID。実行するたびに変わる
```

**手順 (`sample_next`)**

1. `top = scores[:k]` で上位 `k` 個を取り出す
2. `top` のスコアを、課題1 の `softmax(..., temperature)` で確率にする
3. `random.choices(ID のリスト, weights=確率のリスト)[0]` で1つ選んで返す

**ヒント**: `random.choices` は「重み付きのくじ引き」。第7回の `pick_next` と同じ形。

**つまずきやすい点**

- 「トークン ID (int) を返すこと」→ `[0]` を付け忘れて、リストのまま返している
- 「k が効いていない」→ 手順1 を飛ばして、全部の候補から選んでいる
- 「低温 … 種 / 高温 … 種」→ 温度を `softmax` に渡していない

### 例題

次のセルを実行して、動きを見る。

In [ ]:
# 例題
import random

scores = next_scores(encode('日本の首都は'))
top5 = scores[:5]
ids = [i for i, _ in top5]
probs = softmax([s for _, s in top5], 0.8)
print(ids)                                    # [100562, 3583, ...]
print(probs)                                  # [0.61, 0.20, ...]
print(random.choices(ids, weights=probs)[0])

温度の効き方はこうなる。

In [ ]:
# 例題
print(softmax([3.0, 1.0], 0.1))    # [1.00, 0.00]
print(softmax([3.0, 1.0], 2.0))    # [0.73, 0.27]

In [ ]:
import random


def sample_next(scores, temperature=0.8, k=50):
    # TODO 1: 上位 k 個を top にする
    # TODO 2: top のスコアを softmax(..., temperature) で確率にする
    # TODO 3: random.choices で ID を1つ選んで return する
    pass


def generate(prompt, max_new_tokens=60, temperature=0.8, k=50):
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        next_id = sample_next(next_scores(ids), temperature, k)
        if next_id == tok.eos_token_id:
            break
        ids.append(next_id)
    return tok.decode(ids, skip_special_tokens=True)


for t in [0.3, 0.8, 1.5]:
    print(f'--- temperature = {t} ---')
    print(generate('プログラミングを学ぶと', max_new_tokens=40, temperature=t))
    print()

In [ ]:
# [CHECK] q3
need('sample_next', arity=3)

scores = next_scores(encode('日本の首都は'))

# k=1 なら greedy と一致するはず
for _ in range(5):
    assert sample_next(scores, temperature=1.0, k=1) == scores[0][0]
assert isinstance(sample_next(scores, 1.0, 50), int), 'トークン ID (int) を返すこと'

# 候補は必ず上位 k 個の中から出る
top50 = {i for i, _ in scores[:50]}
for _ in range(30):
    assert sample_next(scores, 1.0, 50) in top50
top5 = {i for i, _ in scores[:5]}
for _ in range(30):
    assert sample_next(scores, 1.0, 5) in top5, 'k が効いていない'
for _ in range(30):
    assert sample_next([(7, 0.0), (8, 0.0), (9, 0.0)], 1.0, 2) in (7, 8), 'k が効いていない'

# 温度を上げれば、選ばれるトークンの種類が増えるはず
cold = {sample_next(scores, 0.1, 50) for _ in range(30)}
hot  = {sample_next(scores, 2.0, 50) for _ in range(30)}
assert len(hot) > len(cold), f'低温 {len(cold)} 種 / 高温 {len(hot)} 種'

# 生成が動くこと
g = generate('むかしむかし', max_new_tokens=10, temperature=0.8)
assert isinstance(g, str) and g.startswith('むかしむかし')
print('OK')

---

### `sample_next_top_p` の例題

確率の高い順に足していき、**足す前の合計**が `p` 未満の間だけ候補に残す。

```
出力例 (p = 0.9):
0 足す前の合計 0.00 → 残す
1 足す前の合計 0.50 → 残す
2 足す前の合計 0.80 → 残す
3 足す前の合計 0.95 → 捨てる
```

In [ ]:
# 例題
probs = [0.5, 0.3, 0.15, 0.05]
cum = 0.0
for i, pr in enumerate(probs):
    print(i, f'足す前の合計 {cum:.2f}', '→', '残す' if cum < 0.9 else '捨てる')
    cum += pr

足した後の合計ではなく、**足す前の合計**と比べるのがコツ。
こうすると最上位は必ず残る (足す前は 0) ので、候補が空にならない。

**手順 (`sample_next_top_p`)**

1. 全部のスコアを `softmax(..., temperature)` で確率にする
2. 確率を高い順に見ていき、足す前の合計が `p` 以上になったら止める。それまでの個数を `keep` とする
3. 上位 `keep` 個の中から、`random.choices` で1つ選んで返す

**つまずきやすい点**

- `p` がとても小さいときの確認で止まる → 足した後の合計で比べていて、候補が0個になっている
- 「確率が等しい4つ、p=0.5」の確認で止まる → 足す前の合計が 0.0 と 0.25 の2個は残し、0.5 からは捨てる

In [ ]:
def sample_next_top_p(scores, temperature=0.8, p=0.9):
    # TODO 1: 全部のスコアを softmax(..., temperature) で確率にする
    # TODO 2: 足す前の合計が p に届くまでの個数 keep を数える
    # TODO 3: 上位 keep 個から random.choices で ID を1つ選んで return する
    pass

In [ ]:
# [CHECK] q3p
need('sample_next_top_p', arity=3)

scores = next_scores(encode('日本の首都は'))

# p が極小なら、実質 greedy
for _ in range(5):
    assert sample_next_top_p(scores, 1.0, 1e-6) == scores[0][0]

# p=0.9 の候補は、累積 0.9 に届くまでの語だけ
import math as _m
_mx = scores[0][1]
_es = [_m.exp(s - _mx) for _, s in scores]
allowed, cum = set(), 0.0
for (i, _), e in zip(scores, _es):
    if cum < 0.9:
        allowed.add(i)
    cum += e / sum(_es)
for _ in range(30):
    assert sample_next_top_p(scores, 1.0, 0.9) in allowed

# 確率が等しい4つ、p=0.5 なら上位2つだけ
for _ in range(30):
    assert sample_next_top_p([(1, 0.0), (2, 0.0), (3, 0.0), (4, 0.0)], 1.0, 0.5) in (1, 2)
print('OK')

---
## 課題4: chat (10点)

**やること**: 会話 (発言のリスト) を受け取り、AI の返事だけを文字列で返す `chat(messages)` を書く。

`messages` は `{'role': 話し手, 'content': 発言}` のリスト。
`role` は `'system'` (AI への指示)、`'user'` (人)、`'assistant'` (AI) のどれか。

**入出力の例** (返事は一例)

```
chat([{'role': 'user', 'content': '日本の首都はどこですか？'}]) → '東京です。'
```

**手順** (プロンプトを作る所と、最後の `return` は用意してある。書くのはループ)

1. `for` 文で `max_new_tokens` 回くり返す
2. `sample_next(next_scores(ids), temperature, k)` で次の ID を選ぶ
3. その ID が特殊トークン (`tok.all_special_ids` の中) なら `break`
4. そうでなければ `ids` の末尾に足す

**ヒント**: 課題3 の `generate` とほとんど同じ。止まる条件だけが違う。
特殊トークンは、発言の区切りなどを表す印 (`<|im_end|>` など)。

**つまずきやすい点**

- 「プロンプトが混ざっている」→ `start` を書き換えた、または `ids[start:]` 以外を文字列に戻している
- 「特殊トークンが残っている」→ 手順3 の判定が抜けている
- 「… トークン返った」→ ループの回数が `max_new_tokens` を超えている

### 例題

次のセルを実行して、動きを見る。
まず、`apply_chat_template` (会話を1つの文字列に並べる関数) が何を作るのかを見る。

In [ ]:
# 例題
p = tok.apply_chat_template(
    [{'role': 'user', 'content': 'やあ'}],
    tokenize=False, add_generation_prompt=True)
print(repr(p))
# '<|im_start|>system\n...<|im_end|>\n<|im_start|>user\nやあ<|im_end|>\n
#  <|im_start|>assistant\n'


つまり `chat` は「この文字列の続きを生成する」だけ。
生成のループは課題3 の `generate` とほぼ同じ。違いは、返事の部分だけを取り出すことと、止まる条件。

In [ ]:
# 例題
start = len(ids)
# ... ここで課題3 と同じループ ...
tok.decode(ids[start:], skip_special_tokens=True)

止まる判定は `if next_id in tok.all_special_ids: break`。
`<|im_end|>` も特殊トークンなので、これで応答の終わりを捕まえられる。

In [ ]:
def chat(messages, max_new_tokens=60, temperature=0.7, k=50):
    prompt = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    ids = encode(prompt)
    start = len(ids)
    # TODO 1: for 文で max_new_tokens 回くり返す
    # TODO 2: sample_next(next_scores(ids), temperature, k) で next_id を選ぶ
    # TODO 3: next_id が tok.all_special_ids の中なら break
    # TODO 4: ids の末尾に next_id を足す
    return tok.decode(ids[start:], skip_special_tokens=True)


msgs = [
    {'role': 'system', 'content': 'あなたは簡潔に答えるアシスタントです。'},
    {'role': 'user', 'content': '日本の首都はどこですか？'},
]
print(chat(msgs))

In [ ]:
# [CHECK] q4
need('chat', arity=1)

reply = chat([{'role': 'user', 'content': '1 + 1 は？'}], max_new_tokens=30)
assert isinstance(reply, str) and reply.strip(), '空の応答'
assert '1 + 1 は？' not in reply, 'プロンプトが混ざっている (start の位置を見直す)'
assert '<|im_start|>' not in reply and '<|im_end|>' not in reply, '特殊トークンが残っている'
assert 'system' not in reply.lower()[:20], 'テンプレート全体を返している'

# max_new_tokens を守ること
short = chat([{'role': 'user', 'content': '長い昔話をして'}], max_new_tokens=5)
assert len(tok(short).input_ids) <= 6, f'{len(tok(short).input_ids)} トークン返った'
print('OK')


### 課題4-b: 会話を続ける (5点)

**やること**: 人の発言のリスト `user_lines` を順に `chat` に渡し、AI の返事のリストを返す
`chat_session(user_lines)` を書く。そのとき、それまでの会話 (履歴) を毎回すべて渡す。

**入出力の例** (返事は一例。返すリストの長さは入力と同じ)

```
chat_session(['こんにちは', '元気？']) → ['こんにちは！', '元気です。']
```

**手順** (ループを書く)

1. `for u in user_lines:` でくり返す
2. 履歴 `history` に `{'role': 'user', 'content': u}` を足す
3. `r = chat(history, max_new_tokens=max_new_tokens)` で返事を得る
4. 履歴に `{'role': 'assistant', 'content': r}` を足し、`replies` にも `r` を足す

**ヒント**: モデルは前の会話を覚えていない。毎回履歴を全部渡すので、会話が続いているように見える。

**採点できるように `input()` は使わないこと。** 自分で対話したいときは、下の `chat_interactive()` のセルを使う。

**つまずきやすい点**

- 「chat が … 回しか呼ばれていない」→ ループの中で `chat` を呼んでいない
- 確認セルの `seen[...]` の行で止まる → 手順2 か手順4 で履歴に足し忘れている。または system を毎回足している

### 例題

次のセルを実行して、動きを見る。
履歴に、人の発言と AI の返事を交互に足していくだけ。

In [ ]:
# 例題
history = [{'role': 'system', 'content': '簡潔に答えてください。'}]

history.append({'role': 'user', 'content': 'こんにちは'})
r = chat(history)
history.append({'role': 'assistant', 'content': r})

history.append({'role': 'user', 'content': '元気？'})
r2 = chat(history)

課題は、これを `for u in user_lines:` のループにしたもの。
`replies` には返事 `r` だけを足していく。

In [ ]:
SYSTEM = '簡潔に答えてください。'


def chat_session(user_lines, max_new_tokens=40):
    history = [{'role': 'system', 'content': SYSTEM}]
    replies = []
    # TODO 1: for u in user_lines: でくり返す
    # TODO 2: history に user の発言 u を足す
    # TODO 3: r = chat(history, max_new_tokens=max_new_tokens) で返事を得る
    # TODO 4: history に assistant の返事 r を足し、replies にも r を足す
    return replies


for u, r in zip(['犬と猫はどちらが好き？', 'なぜ？', '他には？'],
                chat_session(['犬と猫はどちらが好き？', 'なぜ？', '他には？'])):
    print('あなた:', u)
    print('AI    :', r)
    print()


In [ ]:
# 対話用 (採点しない)。実行すると入力待ち
def chat_interactive():
    history = [{'role': 'system', 'content': SYSTEM}]
    while True:
        user = input('あなた: ')
        if user.strip() in ('quit', 'exit', ''):
            break
        history.append({'role': 'user', 'content': user})
        reply = chat(history)
        print('AI:', reply)
        history.append({'role': 'assistant', 'content': reply})


# chat_interactive()   ← コメントを外すと対話できる


In [ ]:
# [CHECK] q4b
need('chat_session', arity=1)

rs = chat_session(['こんにちは', '元気？', 'さようなら'], max_new_tokens=20)
assert isinstance(rs, list) and len(rs) == 3, f'{len(rs)} 件返った'
assert all(isinstance(r, str) and r.strip() for r in rs), '空の応答がある'
assert all('<|im_' not in r for r in rs), '特殊トークンが残っている'
assert chat_session([]) == []

# 履歴が積み上がっていることを、渡された messages の長さで確かめる
seen = []
_orig = chat
def _spy(messages, **kw):
    seen.append([m['role'] for m in messages])
    return _orig(messages, **kw)
chat = _spy
try:
    chat_session(['A', 'B', 'C'], max_new_tokens=8)
finally:
    chat = _orig
assert len(seen) == 3, f'chat が {len(seen)} 回しか呼ばれていない'
assert seen[0].count('user') == 1, seen[0]
assert seen[1].count('user') == 2 and seen[1].count('assistant') == 1, seen[1]
assert seen[2].count('user') == 3 and seen[2].count('assistant') == 2, seen[2]
assert all(r[0] == 'system' and r.count('system') == 1 for r in seen), 'system は先頭に1つ'
print('OK')


---

## 課題5: 簡易チャットアプリを作る (出力で確認)

**やること**: 話しかけるたびに返事をし、会話を覚えておく「チャットボット」を作る。
`make_chat()` は、話しかけるための関数 `say(text)` を返す。
履歴 `history` は `make_chat` の中に置いてあり、`say` を呼ぶたびに増えていく。

**入出力の例** (返事は一例)

```
bot = make_chat()
bot('日本の首都は？')   → '東京です。'
bot('そこの人口は？')   → '約1400万人です。'   (履歴があるので「そこ」が東京だと分かる)
```

**手順** (`say` の中。人の発言を履歴に足す所までは用意してある)

1. `reply = chat(history, max_new_tokens=48)` で返事を得る
2. 履歴に `{'role': 'assistant', 'content': reply}` を足す

**採点**: 確認セルは無い。下のセルを実行して、3往復の会話が出力に残っていればよい。

In [ ]:
# 課題5: 簡易チャットアプリを作ろう
def make_chat(system_prompt='簡潔に答えてください。'):
    history = [{'role': 'system', 'content': system_prompt}]

    def say(text, **opts):
        history.append({'role': 'user', 'content': text})
        # TODO 1: chat(history, max_new_tokens=48) で返事を得る
        reply = ...
        # TODO 2: history に {'role': 'assistant', 'content': reply} を足す
        ...
        return reply

    return say


bot = make_chat()
for q in ['日本の首都は？', 'そこの人口は？', '有名な食べ物は？']:
    print('Q:', q)
    print('A:', bot(q))
    print()

---

# 発展課題

余力があれば。課題5 の `make_chat` を良くするための部品を作る。
各部品は確認セルで確かめられ、`score()` の点にも入る。

---

## 発展1: 繰り返しの検出 (5点)

**やること**: 文章の中に、同じ `n` 語のまとまりが2回以上出てきたら `True`、なければ `False` を返す
`has_repeat(text, n)` を書く。

**入出力の例**

```
has_repeat('I like cats. I like cats. Yes', 3) → True   ('I like cats.' が2回)
has_repeat('the quick brown fox', 3)           → False
```

**手順**

1. `text.split()` で単語のリストにする
2. 「見たことがあるまとまり」を入れる集合 `seen = set()` を作る
3. `n` 語ずつのまとまりを順に見て、`seen` にあれば `True` を返す。なければ `seen` に足す
4. 最後まで見つからなければ `False` を返す

**早見表**

```python
words = text.split()          # 文字列 -> 単語のリスト
chunk = tuple(words[i:i + n]) # n 語のまとまり。タプルにすると集合に入れられる
seen = set()                  # 見たことがあるものの集合
chunk in seen                 # 入っているか (True / False)
seen.add(chunk)               # 入れる
```

In [ ]:
def has_repeat(text, n=3):
    # TODO: n 語ずつのまとまりを順に見て、前に出てきたものがあれば True、最後まで無ければ False を返す
    pass


print(has_repeat('I like cats. I like cats. Yes', 3))  # True
print(has_repeat('the quick brown fox', 3))             # False

In [ ]:
# [CHECK] e1
need('has_repeat', arity=2)

assert has_repeat('I like cats. I like cats. Yes', 3)
assert not has_repeat('the quick brown fox', 3)
assert has_repeat('la la la', 1)
assert not has_repeat('', 3)
print('OK')

---

## 発展2: 履歴を短くする (5点)

会話が長くなると、モデルに渡せる長さの上限に近づく。そこで古い発言を捨てる。

**やること**: 2つの関数を書く。

- `estimate_tokens(messages)`: 履歴のトークン数のおおよそを返す。全発言の文字数の合計 ÷ 2 でよい
- `compact(messages, limit)`: 見積もりが `limit` を超えている間、いちばん古い発言 (system の次) を1つずつ捨てる。system は必ず残す

**入出力の例**

```
estimate_tokens([{'role': 'user', 'content': 'あいうえお'}]) → 2 か 3 くらい (1 以上ならよい)
```

**ヒント**: リスト `m` について、`m[:1] + m[2:]` は「2番目を除いたリスト」になる。

In [ ]:
def estimate_tokens(messages):
    # TODO: 全発言の content の文字数を合計し、2 で割った数を返す
    pass


def compact(messages, limit):
    # TODO: estimate_tokens が limit を超えている間、system の次の発言を1つずつ捨てる
    pass


msgs = [
    {'role': 'system', 'content': 'あなたは簡潔に答えるアシスタントです。'},
    {'role': 'user', 'content': 'あ' * 200},
    {'role': 'assistant', 'content': 'い' * 200},
    {'role': 'user', 'content': 'う'},
]
print(len(compact(msgs, 50)))    # 2 (system と最後の発言が残る)

In [ ]:
# [CHECK] e2
need('estimate_tokens', arity=1)
need('compact', arity=2)

assert estimate_tokens([{'role': 'user', 'content': 'あいうえお'}]) >= 1
msgs = [
    {'role': 'system', 'content': 's'},
    {'role': 'user', 'content': 'あ' * 200},
    {'role': 'assistant', 'content': 'い' * 200},
    {'role': 'user', 'content': 'う'},
]
c = compact(msgs, 50)
assert c[0]['role'] == 'system'
assert len(c) < len(msgs)
print('OK')

---

## 発展3: ストリーミング表示 (5点)

ChatGPT などが返事を少しずつ表示するのを「ストリーミング」という。
少しずつ届く文字列の断片を、最後に1つにまとめる部品を作る。

**やること**: 文字列のリスト `pieces` を、1つの文字列につなげて返す `collect(pieces)` を書く。

**入出力の例**

```
collect(['む', 'か', 'し']) → 'むかし'
collect([])                 → ''
```

**ヒント**: `''.join(pieces)`、または `+=` でつなげる。

In [ ]:
def collect(pieces):
    # TODO: pieces の文字列を全部つなげて返す
    pass


print(collect(['む', 'か', 'し']))  # むかし

In [ ]:
# [CHECK] e3
need('collect', arity=1)

assert collect(['む', 'か', 'し']) == 'むかし'
assert collect([]) == ''
assert collect(['a', 'b', 'c']) == 'abc'
print('OK')

---

## 発展4: エフォート (low / medium / high) (5点)

エフォートは「どれだけ丁寧に答えるか」の段階。
low は短く、温度高め (ばらける)。high は長く、温度低め (安定する)。

**やること**: `'low'` / `'medium'` / `'high'` を受け取り、その設定 (辞書) を返す `effort_opts(effort)` を書く。

**入出力の例**

```
effort_opts('high') → {'max_new_tokens': 128, 'temperature': 0.3}
```

**ヒント**: 下の表 `EFFORT` から引くだけ。

In [ ]:
EFFORT = {
    'low':    {'max_new_tokens': 32,  'temperature': 1.0},
    'medium': {'max_new_tokens': 64,  'temperature': 0.8},
    'high':   {'max_new_tokens': 128, 'temperature': 0.3},
}

def effort_opts(effort):
    # TODO: EFFORT から effort の設定を引いて返す
    pass

In [ ]:
# [CHECK] e4
need('EFFORT', kind=dict)

assert set(EFFORT) == {'low', 'medium', 'high'}
need('effort_opts', arity=1)
o = effort_opts('high')
assert 'max_new_tokens' in o and 'temperature' in o
assert effort_opts('high')['max_new_tokens'] > effort_opts('low')['max_new_tokens']
assert effort_opts('low')['temperature'] > effort_opts('high')['temperature']
print('OK')

In [ ]:
# [SKIP] 採点の対象外
score()


---
## 提出

1. 上から全部実行し、出力を残したままにする
2. `ファイル → .ipynb をダウンロード` で保存する
3. 締切までに kibaco に提出する

確認セルが全部 `OK` なら提出してよい。上の `score()` は自分で確かめる用で、提出にはならない。
確認セルを消しても、教員の採点は別の検査で行う。